# 6. Linguistic regression models

## Objective

This notebook treats grammar scoring as supervised regression from transcript-derived numeric features to the continuous grammar label. Constant columns are removed, missing values are median-imputed inside the pipeline, and Ridge standardizes the features before fitting.

Evaluation uses shuffled 5-fold cross-validation with `random_state=42`. Ridge provides a regularized linear reference, while Gradient Boosting tests whether nonlinear feature interactions improve the same linguistic representation. The recorded RMSE and Pearson values below are compared with the mean baseline.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

features = pd.read_csv(
    PROJECT_ROOT + r"\outputs\linguistic_features.csv"
)

X = features.drop(columns=["filename", "label"])
y = features["label"]

# Remove constant features
constant_cols = X.columns[X.nunique() <= 1]
X = X.drop(columns=constant_cols)

print("Removed constant features:", list(constant_cols))
print("X shape:", X.shape)

In [ ]:
#RiDGE

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

pred = cross_val_predict(
    model,
    X,
    y,
    cv=kf
)

rmse = np.sqrt(mean_squared_error(y, pred))
pearson = pearsonr(y, pred)[0]

print("Linguistic + Ridge")
print("RMSE:", rmse)
print("Pearson:", pearson)

In [ ]:
baseline_pred = np.full(len(y), y.mean())

baseline_rmse = np.sqrt(
    mean_squared_error(y, baseline_pred)
)

print("Baseline RMSE:", baseline_rmse)
print("Linguistic Ridge RMSE:", rmse)
print("RMSE improvement:", baseline_rmse - rmse)

In [ ]:
#GRAdIENT BOOST

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gbr", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        random_state=42,
        loss="squared_error"
    ))
])

gbr_pred = cross_val_predict(
    gbr,
    X,
    y,
    cv=kf
)

gbr_rmse = np.sqrt(
    mean_squared_error(y, gbr_pred)
)

gbr_pearson = pearsonr(y, gbr_pred)[0]

print("Linguistic + Gradient Boosting")
print("RMSE:", gbr_rmse)
print("Pearson:", gbr_pearson)

## Interpretation

The recorded original-linguistic Ridge result is RMSE `0.9908950705951046` and Pearson `0.5997339184017166`, compared with the mean baseline RMSE `1.2381936808002134`. The recorded Gradient Boosting result is RMSE `0.9954231149072778` and Pearson `0.5949753795963424`. On this feature set, Ridge is the stronger of the two tested models and remains a useful linear component for later feature expansion.